# Essence Wars: Training a PPO Agent

This notebook demonstrates training a PPO (Proximal Policy Optimization) agent to play Essence Wars.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/essence-wars/blob/main/python/notebooks/02_training_ppo.ipynb)

## Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install essence-wars[train]

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from essence_wars import VectorizedEssenceWars, PyGame
from essence_wars._core import STATE_TENSOR_SIZE, ACTION_SPACE_SIZE

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Observation size: {STATE_TENSOR_SIZE}")
print(f"Action space size: {ACTION_SPACE_SIZE}")

## Define the Neural Network

A simple actor-critic network with shared layers and separate policy/value heads.

In [ ]:
class PPONetwork(nn.Module):
    """Actor-Critic network for PPO with action masking."""
    
    def __init__(self, obs_dim: int = STATE_TENSOR_SIZE, 
                 action_dim: int = ACTION_SPACE_SIZE,
                 hidden_dim: int = 256):
        super().__init__()
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        
        # Policy head (actor)
        self.policy = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
        
        # Value head (critic)
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, obs: torch.Tensor, mask: torch.Tensor):
        """Forward pass returning policy logits and value."""
        features = self.shared(obs)
        
        # Get raw logits and mask invalid actions
        logits = self.policy(features)
        # Set invalid actions to large negative value
        logits = logits.masked_fill(~mask.bool(), float('-inf'))
        
        value = self.value(features)
        return logits, value.squeeze(-1)
    
    def get_action(self, obs: torch.Tensor, mask: torch.Tensor, deterministic: bool = False):
        """Sample action from policy."""
        logits, value = self.forward(obs, mask)
        
        if deterministic:
            action = logits.argmax(dim=-1)
        else:
            probs = F.softmax(logits, dim=-1)
            action = torch.multinomial(probs, 1).squeeze(-1)
        
        log_prob = F.log_softmax(logits, dim=-1)
        action_log_prob = log_prob.gather(-1, action.unsqueeze(-1)).squeeze(-1)
        
        return action, action_log_prob, value

# Create network
network = PPONetwork().to(device)
print(f"Network parameters: {sum(p.numel() for p in network.parameters()):,}")

## PPO Trainer

A minimal PPO implementation with GAE (Generalized Advantage Estimation).

In [ ]:
class PPOTrainer:
    """Minimal PPO trainer."""
    
    def __init__(self, network: PPONetwork, lr: float = 3e-4,
                 gamma: float = 0.99, gae_lambda: float = 0.95,
                 clip_eps: float = 0.2, entropy_coef: float = 0.01,
                 value_coef: float = 0.5, max_grad_norm: float = 0.5):
        self.network = network
        self.optimizer = torch.optim.Adam(network.parameters(), lr=lr)
        
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_eps = clip_eps
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef
        self.max_grad_norm = max_grad_norm
    
    def compute_gae(self, rewards, values, dones, next_value):
        """Compute Generalized Advantage Estimation."""
        advantages = torch.zeros_like(rewards)
        last_gae = 0
        
        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_val = next_value
            else:
                next_val = values[t + 1]
            
            delta = rewards[t] + self.gamma * next_val * (1 - dones[t]) - values[t]
            advantages[t] = last_gae = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
        
        returns = advantages + values
        return advantages, returns
    
    def update(self, batch: dict, num_epochs: int = 4, minibatch_size: int = 256):
        """Perform PPO update."""
        obs = batch['obs']
        actions = batch['actions']
        old_log_probs = batch['log_probs']
        advantages = batch['advantages']
        returns = batch['returns']
        masks = batch['masks']
        
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        total_loss = 0
        num_updates = 0
        
        for _ in range(num_epochs):
            # Shuffle indices
            indices = torch.randperm(len(obs))
            
            for start in range(0, len(obs), minibatch_size):
                end = start + minibatch_size
                mb_idx = indices[start:end]
                
                mb_obs = obs[mb_idx]
                mb_actions = actions[mb_idx]
                mb_old_log_probs = old_log_probs[mb_idx]
                mb_advantages = advantages[mb_idx]
                mb_returns = returns[mb_idx]
                mb_masks = masks[mb_idx]
                
                # Forward pass
                logits, values = self.network(mb_obs, mb_masks)
                log_probs = F.log_softmax(logits, dim=-1)
                action_log_probs = log_probs.gather(-1, mb_actions.unsqueeze(-1)).squeeze(-1)
                
                # Policy loss (clipped)
                ratio = torch.exp(action_log_probs - mb_old_log_probs)
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                
                # Value loss
                value_loss = F.mse_loss(values, mb_returns)
                
                # Entropy bonus
                probs = F.softmax(logits, dim=-1)
                entropy = -(probs * log_probs).sum(dim=-1).mean()
                
                # Total loss
                loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy
                
                # Optimize
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.network.parameters(), self.max_grad_norm)
                self.optimizer.step()
                
                total_loss += loss.item()
                num_updates += 1
        
        return total_loss / num_updates

trainer = PPOTrainer(network)
print("PPO trainer initialized")

## Training Loop

Collect rollouts and train the agent.

In [ ]:
def collect_rollout(env, network, num_steps: int = 128):
    """Collect a rollout from the environment."""
    obs_list, action_list, reward_list = [], [], []
    log_prob_list, value_list, done_list, mask_list = [], [], [], []
    
    obs, masks = env.reset(seed=np.random.randint(0, 2**31))
    
    for _ in range(num_steps):
        obs_t = torch.from_numpy(obs).float().to(device)
        mask_t = torch.from_numpy(masks).float().to(device)
        
        with torch.no_grad():
            actions, log_probs, values = network.get_action(obs_t, mask_t)
        
        # Step environment
        actions_np = actions.cpu().numpy().astype(np.uint8)
        next_obs, rewards, dones, next_masks = env.step(actions_np)
        
        # Store
        obs_list.append(obs_t)
        action_list.append(actions)
        log_prob_list.append(log_probs)
        value_list.append(values)
        reward_list.append(torch.from_numpy(rewards).float().to(device))
        done_list.append(torch.from_numpy(dones.astype(np.float32)).to(device))
        mask_list.append(mask_t)
        
        obs, masks = next_obs, next_masks
    
    # Get final value for GAE
    with torch.no_grad():
        obs_t = torch.from_numpy(obs).float().to(device)
        mask_t = torch.from_numpy(masks).float().to(device)
        _, next_value = network(obs_t, mask_t)
    
    # Stack tensors
    obs_batch = torch.stack(obs_list)
    actions_batch = torch.stack(action_list)
    log_probs_batch = torch.stack(log_prob_list)
    values_batch = torch.stack(value_list)
    rewards_batch = torch.stack(reward_list)
    dones_batch = torch.stack(done_list)
    masks_batch = torch.stack(mask_list)
    
    # Compute advantages
    advantages, returns = trainer.compute_gae(
        rewards_batch.mean(dim=1), values_batch.mean(dim=1),
        dones_batch.mean(dim=1), next_value.mean()
    )
    
    # Flatten batch
    batch_size = num_steps * env.num_envs
    return {
        'obs': obs_batch.view(batch_size, -1),
        'actions': actions_batch.view(batch_size),
        'log_probs': log_probs_batch.view(batch_size),
        'advantages': advantages.repeat_interleave(env.num_envs),
        'returns': returns.repeat_interleave(env.num_envs),
        'masks': masks_batch.view(batch_size, -1),
    }, rewards_batch.sum().item() / env.num_envs

print("Rollout collection function defined")

## Train the Agent

Run training for a specified number of timesteps.

In [ ]:
# Training hyperparameters
NUM_ENVS = 32
NUM_STEPS = 128  # Steps per rollout
TOTAL_TIMESTEPS = 50_000  # Increase for better results
EVAL_INTERVAL = 10_000

# Create vectorized environment
env = VectorizedEssenceWars(num_envs=NUM_ENVS)

# Training loop
timesteps = 0
iteration = 0
rewards_history = []

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"Batch size: {NUM_ENVS * NUM_STEPS:,}")
print()

while timesteps < TOTAL_TIMESTEPS:
    # Collect rollout
    batch, episode_reward = collect_rollout(env, network, NUM_STEPS)
    rewards_history.append(episode_reward)
    
    # Update
    loss = trainer.update(batch)
    
    timesteps += NUM_ENVS * NUM_STEPS
    iteration += 1
    
    # Log progress
    if iteration % 10 == 0:
        mean_reward = np.mean(rewards_history[-10:])
        print(f"Iter {iteration:4d} | Steps: {timesteps:6,} | Loss: {loss:.4f} | Reward: {mean_reward:+.3f}")
    
    # Evaluation
    if timesteps % EVAL_INTERVAL < NUM_ENVS * NUM_STEPS:
        network.eval()
        wins = 0
        eval_games = 20
        
        for seed in range(eval_games):
            game = PyGame(deck1='artificer_tokens', deck2='broodmother_pack')
            game.reset(seed=seed)
            
            while not game.is_done():
                if game.current_player() == 0:
                    obs = torch.from_numpy(game.observe()).float().unsqueeze(0).to(device)
                    mask = torch.from_numpy(game.action_mask()).float().unsqueeze(0).to(device)
                    with torch.no_grad():
                        action, _, _ = network.get_action(obs, mask, deterministic=True)
                    game.step(action.item())
                else:
                    game.step(game.greedy_action())
            
            if game.winner() == 0:
                wins += 1
        
        win_rate = wins / eval_games * 100
        print(f"\n>>> Evaluation: {win_rate:.0f}% win rate vs Greedy ({wins}/{eval_games} games)\n")
        network.train()

env.close()
print("Training complete!")

## Save and Load Model

In [ ]:
# Save checkpoint
checkpoint = {
    'network_state_dict': network.state_dict(),
    'config': {
        'obs_dim': STATE_TENSOR_SIZE,
        'action_dim': ACTION_SPACE_SIZE,
        'hidden_dim': 256,
    },
}
torch.save(checkpoint, 'ppo_quickstart.pt')
print("Model saved to ppo_quickstart.pt")

# Load checkpoint
loaded = torch.load('ppo_quickstart.pt', weights_only=False)
loaded_network = PPONetwork()
loaded_network.load_state_dict(loaded['network_state_dict'])
print("Model loaded successfully")

## Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Smooth rewards with moving average
window = min(20, len(rewards_history) // 5 + 1)
smoothed = np.convolve(rewards_history, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(rewards_history, alpha=0.3, label='Raw')
plt.plot(range(window-1, len(rewards_history)), smoothed, label=f'Smoothed (window={window})')
plt.xlabel('Iteration')
plt.ylabel('Episode Reward')
plt.title('PPO Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Next Steps

- **Increase training time**: 50K steps is just for demonstration. Try 500K+ for better results.
- **Tune hyperparameters**: Learning rate, entropy coefficient, etc.
- **Add curriculum**: Start against random, then greedy, then self-play.
- **Use the CLI**: `essence-wars train ppo --timesteps 500000` for full training.

See [03_evaluation.ipynb](03_evaluation.ipynb) for benchmarking your trained agent.